# Treinamento do Modelo

- Treinar modelo
- Exportar NIR

# Exportar Dataset de Teste

- Salvar dataset como .npz
  - chave 'data' para os dados
  - chave 'labels' para os rótulos

## Exemplo com Torch

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np

# Transformação: converte PIL Images para tensores e normaliza
transform = transforms.Compose([
    transforms.ToTensor(),             # converte para torch.Tensor
    transforms.Normalize((0.1307,), (0.3081,))  # normalização MNIST
])

# Baixa MNIST de treino
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Cria DataLoader
dataloader_torch = DataLoader(mnist_train, batch_size=1, shuffle=True)

In [ ]:
def salvar_dataset_npz(loader, arquivo):
    dados = []
    labels = []

    for batch in loader:
        # formato (x, y)
        x, y = batch

        # PyTorch
        if torch.is_tensor(x):
            x = x.cpu().numpy()
            y = y.cpu().numpy()

        dados.append(x)
        labels.append(y)

    dados = np.concatenate(dados, axis=0)
    labels = np.concatenate(labels, axis=0)

    np.savez_compressed(
        arquivo,
        data=dados,
        labels=labels
    )

    print(f"Arquivo '{arquivo}.npz' salvo.")

In [ ]:
salvar_dataset_npz(dataloader_torch, "data_test.npz")

Arquivo 'data_test.npz.npz' salvo.


## Exemplo com Tensorflow

- TODO: FINALIZAR

In [ ]:
import tensorflow as tf

# Carrega MNIST direto do Keras datasets
(mnist_x_train, mnist_y_train), _ = tf.keras.datasets.mnist.load_data()

# Normaliza os pixels
mnist_x_train = mnist_x_train.astype('float32') / 255.0
mnist_x_train = mnist_x_train[..., tf.newaxis]  # adiciona canal (28,28,1)

# Converte labels para tf.int32
mnist_y_train = mnist_y_train.astype('int32')

# Cria tf.data.Dataset
dataset_tf = tf.data.Dataset.from_tensor_slices((mnist_x_train, mnist_y_train))
dataset_tf = dataset_tf.shuffle(buffer_size=10000).batch(32)

In [ ]:
import numpy as np

def processar_batch(batch_x, batch_y):
    """
    Função genérica para processar um batch de dados.
    Aqui você pode colocar treino, validação, ou qualquer operação.
    """
    print("Batch X shape:", batch_x.shape)
    print("Batch Y shape:", batch_y.shape)
    # Exemplo de operação simples
    return batch_x.mean(), batch_y.mean()

def iterar_dataloader(dataloader):
    """
    Itera sobre qualquer dataloader PyTorch ou TensorFlow,
    convertendo para numpy arrays.
    """
    for batch_x, batch_y in dataloader:
        # PyTorch e TensorFlow 2.x retornam tensors com .numpy()
        if hasattr(batch_x, "numpy"):
            batch_x = batch_x.numpy()
            batch_y = batch_y.numpy()
        # Agora batch_x e batch_y são np.array
        yield batch_x, batch_y
        
def main(dataloader_torch, dataloader_tf):
    print("Iterando DataLoader do PyTorch")
    for batch_x, batch_y in iterar_dataloader(dataloader_torch):
        processar_batch(batch_x, batch_y)
    
    print("\nIterando DataLoader do TensorFlow")
    for batch_x, batch_y in iterar_dataloader(dataloader_tf):
        processar_batch(batch_x, batch_y)

In [ ]:
main(dataloader_torch, dataset_tf)

# NeuroHls

In [1]:
from NeuroHls import *

In [ ]:
neuro_hls = NeuroHls("z_test", should_recreate_files=False)

## Definindo a Implementação do Modelo

In [ ]:
nir_file = "dense_only_linear1.nir"
model_config = neuro_hls.read_nir_file(nir_file)

In [6]:
print(model_config)

------------------------------
Layer 1: Dense (784, 256)
------------------------------

	Unroll Factors:
		- Accum: 1
		- Fire: 1
	Quantization (ap_fixed<16, 8>):
		- Total bits: 16
		- Integer bits: 8
		- Fractional bits: 8

------------------------------
Layer 2: Dense (256, 128)
------------------------------

	Unroll Factors:
		- Accum: 1
		- Fire: 1
	Quantization (ap_fixed<16, 8>):
		- Total bits: 16
		- Integer bits: 8
		- Fractional bits: 8

------------------------------
Layer 3: Dense (128, 10)
------------------------------

	Unroll Factors:
		- Accum: 1
		- Fire: 1
	Quantization (ap_fixed<16, 8>):
		- Total bits: 16
		- Integer bits: 8
		- Fractional bits: 8



In [5]:
model_config.layers[0].set_unroll_factors(10, 8)
model_config.layers[0].set_potential_quantization(32, 12)

The accumulation unroll factor must be a divisor of the number of inputs (784). Used value: 8
Used fire unroll factor: 8


In [6]:
print(model_config)

------------------------------
Layer 1: Dense (784, 128)
------------------------------

	Unroll Factors:
		- Accum: 8
		- Fire: 8
	Quantization (ap_fixed<32, 12>):
		- Total bits: 32
		- Integer bits: 12
		- Fractional bits: 20

------------------------------
Layer 2: Dense (128, 10)
------------------------------

	Unroll Factors:
		- Accum: 1
		- Fire: 1
	Quantization (ap_fixed<16, 8>):
		- Total bits: 16
		- Integer bits: 8
		- Fractional bits: 8



In [7]:
neuro_hls.implement_model_from_config(model_config)

## Criando o Testbench

In [ ]:
neuro_hls.define_test_dataset("data_test.npz", data_is_binary=False, step_count=10, different_sample_per_step=False)

In [ ]:
neuro_hls.create_testbench(total_samples=500, batch_size=30)

Testbench Criado


In [ ]:
neuro_hls.run_csim()